In [3]:
# ============================================================
# SevaSetu - Deep Learning NLP Department Classifier
# Model: Embedding + Bidirectional LSTM
# Dataset: 100,000 grievance complaints
# ============================================================

import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = "grievance_dataset.csv"

MODEL_PATH = "sevasetu_department_bilstm.keras"
LABEL_ENCODER_PATH = "department_label_encoder.pkl"
VOCAB_PATH = "sevasetu_vocabulary.pkl"

MAX_VOCAB_SIZE = 30000
MAX_SEQUENCE_LENGTH = 150

EMBEDDING_DIM = 128
LSTM_UNITS = 128

BATCH_SIZE = 128
EPOCHS = 15

RANDOM_STATE = 42


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3. KEEP ONLY REQUIRED COLUMNS
# ============================================================

data = df[["complaint_text", "department"]].copy()

# Remove missing values
data.dropna(subset=["complaint_text", "department"], inplace=True)

# Convert to string
data["complaint_text"] = data["complaint_text"].astype(str)
data["department"] = data["department"].astype(str)


# ============================================================
# 4. TEXT CLEANING
# ============================================================

def clean_text(text):
    """
    Basic text cleaning.

    We intentionally do not remove all punctuation or numbers
    because some complaint information may be useful.
    """

    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("\nCleaning text...")

data["complaint_text"] = data["complaint_text"].apply(clean_text)


# ============================================================
# 5. REMOVE DUPLICATE ROWS
# ============================================================

data.drop_duplicates(
    subset=["complaint_text", "department"],
    inplace=True
)

print("\nDataset after cleaning:", data.shape)


# ============================================================
# 6. ENCODE DEPARTMENT LABELS
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(data["department"])

num_classes = len(label_encoder.classes_)

print("\nNumber of departments:", num_classes)

print("\nDepartment classes:")

for i, department in enumerate(label_encoder.classes_):
    print(i, "->", department)


# ============================================================
# 7. INPUT DATA
# ============================================================

X = data["complaint_text"].values


# ============================================================
# 8. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First split:
# 80% train
# 20% temporary

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Second split:
# 10% validation
# 10% test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nData split:")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


# ============================================================
# 9. TEXT VECTORIZATION
# ============================================================

print("\nCreating TextVectorization layer...")

vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

# Learn vocabulary ONLY from training data
vectorizer.adapt(X_train)

vocab_size = len(vectorizer.get_vocabulary())

print("Vocabulary size:", vocab_size)


# ============================================================
# 10. CREATE TF.DATA DATASETS
# ============================================================

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test)
)


train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# ============================================================
# 11. BUILD BiLSTM MODEL
# ============================================================

print("\nBuilding BiLSTM model...")

model = Sequential([

    # Convert text -> integer token IDs
    vectorizer,

    # Word embeddings
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),

    # Bidirectional LSTM
    Bidirectional(
        LSTM(
            LSTM_UNITS,
            return_sequences=False
        )
    ),

    # Regularization
    Dropout(0.40),

    # Dense layer
    Dense(
        128,
        activation="relu"
    ),

    BatchNormalization(),

    Dropout(0.30),

    # Output layer
    Dense(
        num_classes,
        activation="softmax"
    )
])


# ============================================================
# 12. COMPILE MODEL
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)


# ============================================================
# 13. MODEL SUMMARY
# ============================================================

model.summary()


# ============================================================
# 14. CALLBACKS
# ============================================================

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]


# ============================================================
# 15. TRAIN MODEL
# ============================================================

print("\nStarting training...\n")

history = model.fit(
    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks
)


# ============================================================
# 16. EVALUATE MODEL
# ============================================================




# ============================================================
# 20. SAVE LABEL ENCODER
# ============================================================

with open(
    LABEL_ENCODER_PATH,
    "wb"
) as f:

    pickle.dump(
        label_encoder,
        f
    )


# ============================================================
# 21. SAVE VOCABULARY
# ============================================================

vocabulary = vectorizer.get_vocabulary()

with open(
    VOCAB_PATH,
    "wb"
) as f:

    pickle.dump(
        vocabulary,
        f
    )








Loading dataset...
Dataset shape: (100000, 7)

Columns:
['complaint_id', 'sector', 'complaint_text', 'location', 'complaint_type', 'phone_number', 'department']

Cleaning text...

Dataset after cleaning: (97482, 2)

Number of departments: 12

Department classes:
0 -> Department of Food & Civil Supplies
1 -> Department of Health & Family Welfare
2 -> Department of School Education
3 -> Department of Social Welfare
4 -> Municipal Corporation - Sanitation
5 -> Police Department
6 -> Public Works Department (PWD)
7 -> Revenue Department
8 -> State Electricity Board
9 -> State Pollution Control Board
10 -> State Transport Corporation
11 -> Water Board

Data split:
Training: 77985
Validation: 9748
Testing: 9749

Creating TextVectorization layer...
Vocabulary size: 367

Building BiLSTM model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting training...

Epoch 1/15


c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 17/610 ━━━━━━━━━━━━━━━━━━━━ 3:52 392ms/step - accuracy: 0.2068 - loss: 2.3342

KeyboardInterrupt: 

In [4]:
from tensorflow.keras.models import load_model

print("\nEvaluating model...")

model=load_model("sevasetu_department_bilstm.keras")

test_loss, test_accuracy = model.evaluate(
    test_dataset
)

print("\n========================================")
print("TEST RESULTS")
print("========================================")

print("Test Loss     :", test_loss)
print("Test Accuracy :", test_accuracy)


# ============================================================
# 17. GENERATE PREDICTIONS
# ============================================================

print("\nGenerating predictions...")

test_inputs = tf.constant(
    X_test.tolist(),
    dtype=tf.string
)

probabilities = model.predict(
    test_inputs,
    batch_size=BATCH_SIZE,
    verbose=1
)

predictions = np.argmax(
    probabilities,
    axis=1
)


# ============================================================
# 18. CLASSIFICATION REPORT
# ============================================================

print("\n========================================")
print("CLASSIFICATION REPORT")
print("========================================")

print(
    classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_,
        digits=4
    )
)


# ============================================================
# 19. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    predictions
)

print("\n========================================")
print("CONFUSION MATRIX")
print("========================================")

print(cm)
# ============================================================
# LOAD MODEL
# ============================================================

model = load_model(
    "sevasetu_department_bilstm.keras"
)


# ============================================================
# LOAD LABEL ENCODER
# ============================================================

with open(
    "department_label_encoder.pkl",
    "rb"
) as f:
    label_encoder = pickle.load(f)


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_department(complaint):

    # Make sure complaint is a string
    complaint = str(complaint)

    # Create TensorFlow tensor
    input_text = tf.constant(
        [complaint],
        dtype=tf.string
    )

    # Predict
    probabilities = model.predict(
        input_text,
        verbose=0
    )[0]

    # Highest probability
    predicted_index = int(
        np.argmax(probabilities)
    )

    # Convert index -> department name
    department = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    # Confidence
    confidence = float(
        probabilities[predicted_index]
    )

    return {
        "department": department,
        "confidence": confidence
    }
    
    print("\n========================================")
print("UNSEEN COMPLAINT TEST")
print("========================================")


real_world_tests = [

    # "The electricity transformer near our school exploded yesterday and the entire locality is without power.",

    # "My family has not received ration from the government fair price shop for the last three months.",

    # "The road connecting our village to the main highway has developed huge potholes and is almost impossible to use.",

    # "The hospital in our area does not have enough doctors and patients are waiting for hours.",

    # "A person stole my motorcycle yesterday and I want to register a police complaint.",

    # "The drainage system is overflowing and dirty water is entering houses.",

    # "There is a large amount of industrial waste being dumped into the river near our village.",

    # "I applied for correction of my land ownership record but the revenue office has not processed it.",

    # "The government bus service to our village has stopped operating.",

    # "There is no proper drinking water supply in our locality for the last week.",
    "There is maar pit in my area one person has shoot the other person he is dead on spot"
]


for complaint in real_world_tests:

    result = predict_department(
        complaint
    )

    print("\nComplaint:")
    print(complaint)

    print("Predicted Department:")
    print(result["department"])

    print(
        "Confidence:",
        f"{result['confidence']:.4f}"
    )


Evaluating model...
77/77 ━━━━━━━━━━━━━━━━━━━━ 10s 121ms/step - accuracy: 1.0000 - loss: 2.5782e-04

TEST RESULTS
Test Loss     : 0.00025782256852835417
Test Accuracy : 1.0

Generating predictions...
77/77 ━━━━━━━━━━━━━━━━━━━━ 9s 113ms/step

CLASSIFICATION REPORT
                                       precision    recall  f1-score   support

  Department of Food & Civil Supplies     1.0000    1.0000    1.0000       820
Department of Health & Family Welfare     1.0000    1.0000    1.0000       812
       Department of School Education     1.0000    1.0000    1.0000       811
         Department of Social Welfare     1.0000    1.0000    1.0000       814
   Municipal Corporation - Sanitation     1.0000    1.0000    1.0000       805
                    Police Department     1.0000    1.0000    1.0000       812
        Public Works Department (PWD)     1.0000    1.0000    1.0000       804
                   Revenue Department     1.0000    1.0000    1.0000       817
              State Ele